# Training the GST Regulatory LLM -- Colab

Trains the 131.5M param model on the GST regulatory corpus, pulled from
Hugging Face (`Tharun007/gst-rulings-corpus`) via `huggingface_hub` --
same source and verification as the Kaggle notebook. No Kaggle API
credentials needed here.

**Colab's local disk does NOT survive a session disconnect.** Only
checkpoints are stored on mounted Google Drive, since losing hours of
training progress is the expensive failure mode. The dataset itself is
small (~10.6M tokens) and downloads fresh from HF in seconds each
session -- caching it in Drive isn't worth the quota or the stale-cache
risk, so it lives on local disk instead.


In [ ]:
!pip install -q torch numpy tiktoken pandas pyarrow huggingface_hub

In [ ]:
# ---- Config -- EDIT THIS ----
CONTEXT_LEN = 1024
BATCH_SIZE = 8
MAX_STEPS = 20000
EVAL_EVERY = 250
EVAL_ITERS = 50
LR = 3e-4
WEIGHT_DECAY = 0.1
PATIENCE = 10  # early-stopping: rounds with no val improvement before stopping

# ---- HF dataset location -- EDIT IF THE REPO/FILENAMES CHANGE ----
HF_REPO_ID = "Tharun007/gst-rulings-corpus"
HF_TRAIN_FILE = "data/train-00000-of-00001.parquet"
HF_VAL_FILE = "data/validation-00000-of-00001.parquet"
HF_TOKEN = None  # repo is public -- set a token string only if you make it private


## Mount Drive for checkpoint persistence

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/llm-from-scratch/checkpoints"
DATA_CACHE_DIR = "/content/data"  # local disk -- small corpus, cheap to re-fetch each session

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DATA_CACHE_DIR, exist_ok=True)
print(f"Checkpoints -> {CHECKPOINT_DIR}")
print(f"Data cache  -> {DATA_CACHE_DIR}")


## Download corpus from Hugging Face

Uses `hf_hub_download` (not the Kaggle API, and not a raw `requests.get`
on a hand-built URL -- that combination is what produced an HTML-saved-
as-.parquet failure earlier in this project). Each downloaded file's
magic bytes are checked before pandas touches it, so a bad fetch throws a
clear error here instead of a confusing `ArrowInvalid` three cells later.

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

def _verify_parquet_magic(path):
    """Parquet files start and end with the 4 bytes b'PAR1'. A bad fetch
    (HTML error page, truncated download, etc.) won't have this -- catch
    it here instead of a confusing ArrowInvalid from pandas later."""
    with open(path, "rb") as f:
        head = f.read(4)
        f.seek(-4, os.SEEK_END)
        tail = f.read(4)
    if head != b"PAR1" or tail != b"PAR1":
        preview = open(path, "rb").read(300)
        raise ValueError(
            f"{path} is not a valid parquet file (header={head!r}, footer={tail!r}). "
            f"First 300 bytes:\n{preview}"
        )

def _download_and_verify(filename):
    local_path = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN,
        cache_dir=DATA_CACHE_DIR,
    )
    _verify_parquet_magic(local_path)
    return local_path

train_path = _download_and_verify(HF_TRAIN_FILE)
val_path = _download_and_verify(HF_VAL_FILE)
print(f"Train file -> {train_path}")
print(f"Val file   -> {val_path}")

train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)

assert "text" in train_df.columns, f"Expected a 'text' column, got {list(train_df.columns)}"
print(f"Train rows: {len(train_df):,} | Val rows: {len(val_df):,}")
print("Sample:", train_df["text"].iloc[0][:200])


## Tokenize + pack

In [ ]:
import numpy as np
import tiktoken

enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token

def tokenize_and_pack(texts, context_len):
    all_ids = []
    for t in texts:
        all_ids.extend(enc.encode(t))
        all_ids.append(EOT)
    arr = np.array(all_ids, dtype=np.uint16)
    n_seq = len(arr) // context_len
    return arr[: n_seq * context_len].reshape(n_seq, context_len)

train_data = tokenize_and_pack(train_df["text"].tolist(), CONTEXT_LEN)
val_data = tokenize_and_pack(val_df["text"].tolist(), CONTEXT_LEN)
print(f"Train sequences: {train_data.shape[0]:,}")
print(f"Val sequences:   {val_data.shape[0]:,}")


## Model architecture

In [ ]:
"""
Model architecture -- GST/tax regulatory LLM project.

Config locked earlier in this project: d_model=768, n_layers=13, n_heads=12,
vocab_size=50257 (GPT-2 BPE), context_length=1024, weight-tied embeddings.
~131.3M parameters -- one layer deeper than GPT-2 small (124M), landing on
the target 130M while staying inside the proven d/n aspect-ratio band
(ratio ~59, same GPT-2 hyperparameter conventions).

Architecture follows the standard GPT-2-style decoder-only transformer
(as in Raschka's book): token + positional embeddings -> N transformer
blocks (pre-LayerNorm, multi-head causal self-attention, GELU feedforward)
-> final LayerNorm -> weight-tied output head.

Dropout defaults to 0.2 (not the book's 0.1) -- deliberately higher given
this project's real corpus size (~10.6M tokens) is far below the
Chinchilla-optimal budget for this parameter count; heavier regularization
is load-bearing here, not optional.
"""

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GPTConfig:
    vocab_size: int = 50257
    context_length: int = 1024
    d_model: int = 768
    n_layers: int = 13
    n_heads: int = 12
    dropout: float = 0.2
    qkv_bias: bool = False


class MultiHeadAttention(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.d_model = cfg.d_model

        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=cfg.qkv_bias)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)

        # Causal mask -- precomputed once, not learned. Registered as a
        # buffer so it moves with .to(device) automatically but isn't
        # saved as a trainable parameter.
        mask = torch.triu(torch.ones(cfg.context_length, cfg.context_length), diagonal=1).bool()
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        qkv = self.qkv(x)  # (B, T, 3*d_model)
        q, k, v = qkv.split(self.d_model, dim=2)

        # (B, T, n_heads, head_dim) -> (B, n_heads, T, head_dim)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn_scores = attn_scores.masked_fill(self.causal_mask[:T, :T], float("-inf"))
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        out = attn_weights @ v  # (B, n_heads, T, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.out_proj(out))
        return out


class FeedForward(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.d_model, 4 * cfg.d_model),
            nn.GELU(),
            nn.Linear(4 * cfg.d_model, cfg.d_model),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerBlock(nn.Module):
    """Pre-LayerNorm block: LN -> attention -> residual, LN -> FFN -> residual."""

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = MultiHeadAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg

        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.context_length, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)

        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_final = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # Weight tying: share the token embedding matrix with the output
        # head. Saves ~38.6M params at this vocab size/d_model with no
        # real capacity loss.
        self.head.weight = self.token_emb.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None):
        B, T = idx.shape
        assert T <= self.cfg.context_length, (
            f"sequence length {T} exceeds context_length {self.cfg.context_length}"
        )

        pos = torch.arange(T, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)

        for block in self.blocks:
            x = block(x)

        x = self.ln_final(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1
            )

        return logits, loss

    def num_params(self, exclude_embeddings: bool = False) -> int:
        n = sum(p.numel() for p in self.parameters())
        if exclude_embeddings:
            n -= self.token_emb.weight.numel()  # tied, so this also covers the head
        return n

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int, temperature: float = 1.0,
                 top_k: int = None):
        """Simple autoregressive sampling -- greedy if temperature=0, else
        multinomial sampling with optional top-k filtering."""
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.context_length:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")

            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx


In [ ]:
cfg = GPTConfig(context_length=CONTEXT_LEN)
model = GPTModel(cfg)
print(f"Total parameters: {model.num_params():,}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
model = model.to(device)


## Training loop

In [ ]:
def get_batch(data, batch_size, device):
    idx = np.random.randint(0, data.shape[0], size=batch_size)
    seqs = torch.from_numpy(data[idx].astype(np.int64))
    x = seqs[:, :-1].contiguous()
    y = seqs[:, 1:].contiguous()
    return x.to(device), y.to(device)


def configure_optimizer(model, weight_decay, lr):
    decay_params, no_decay_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        (decay_params if param.dim() >= 2 else no_decay_params).append(param)
    return torch.optim.AdamW([
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ], lr=lr, betas=(0.9, 0.95))


@torch.no_grad()
def estimate_loss(model, data, batch_size, device, eval_iters=50):
    model.eval()
    losses = torch.zeros(eval_iters)
    for i in range(eval_iters):
        x, y = get_batch(data, batch_size, device)
        _, loss = model(x, y)
        losses[i] = loss.item()
    model.train()
    return losses.mean().item()


def save_checkpoint(path, model, optimizer, step, best_val_loss, no_improve_count, cfg):
    # no_improve_count is saved here -- the original version of this notebook
    # dropped it, which silently reset your early-stopping patience counter
    # to 0 on every resume. Fixed.
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "step": step,
        "best_val_loss": best_val_loss,
        "no_improve_count": no_improve_count,
        "torch_rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "config": vars(cfg),
    }, path)


In [ ]:
# Resume support -- if a checkpoint already exists in CHECKPOINT_DIR (e.g.
# from a previous session that got cut off), pick up from there instead
# of starting over.
import os as _os

last_ckpt_path = _os.path.join(CHECKPOINT_DIR, "last.pt")
optimizer = configure_optimizer(model, WEIGHT_DECAY, LR)

start_step = 0
best_val_loss = float("inf")
no_improve_count = 0

if _os.path.exists(last_ckpt_path):
    print(f"Found existing checkpoint at {last_ckpt_path} -- resuming.")
    ckpt = torch.load(last_ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_step = ckpt["step"]
    best_val_loss = ckpt["best_val_loss"]
    no_improve_count = ckpt.get("no_improve_count", 0)  # .get() for backward compat
    if "torch_rng_state" in ckpt:
        torch.set_rng_state(ckpt["torch_rng_state"].to(torch.uint8).cpu())
    if "numpy_rng_state" in ckpt:
        np.random.set_state(ckpt["numpy_rng_state"])
    print(f"  Resumed at step {start_step}, best_val_loss so far: {best_val_loss:.4f}, "
          f"no_improve_count: {no_improve_count}")
else:
    print("No existing checkpoint -- starting fresh.")


In [ ]:
import time

model.train()
t0 = time.time()

for step in range(start_step, MAX_STEPS):
    x, y = get_batch(train_data, BATCH_SIZE, device)
    _, loss = model(x, y)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if step % EVAL_EVERY == 0 or step == MAX_STEPS - 1:
        train_loss_est = estimate_loss(model, train_data, BATCH_SIZE, device, EVAL_ITERS)
        val_loss = estimate_loss(model, val_data, BATCH_SIZE, device, EVAL_ITERS)
        elapsed = time.time() - t0
        print(f"step {step:6d} | train_loss {train_loss_est:.4f} | "
              f"val_loss {val_loss:.4f} | {elapsed:.0f}s elapsed")

        save_checkpoint(last_ckpt_path, model, optimizer, step, best_val_loss,
                         no_improve_count, cfg)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve_count = 0
            save_checkpoint(_os.path.join(CHECKPOINT_DIR, "best.pt"),
                             model, optimizer, step, best_val_loss, no_improve_count, cfg)
            print(f"  New best val_loss: {best_val_loss:.4f} -- saved best.pt")
        else:
            no_improve_count += 1
            print(f"  No improvement ({no_improve_count}/{PATIENCE})")

        if no_improve_count >= PATIENCE:
            print(f"\nEarly stopping at step {step} -- best.pt is your model, not last.pt.")
            break

print("\nTraining complete (or interrupted -- re-run this cell to resume from last.pt).")
